# SOLUTIONS · Laptop lab 3: Linear classification and preprocessing
*Build a pipeline that spots the premium listings*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 12, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you met linear classifiers: logistic regression and the linear SVM, with their penalty `C` (Lecture 5). And you met preprocessing: scaling, one-hot encoding, new features, and the pipeline that keeps all of it inside cross-validation (Lecture 6). Today you use both on the Airbnb listings of Labs 1 and 2.

**The story.** This time the question is not *how much* a listing costs, but *whether* it is a **premium** listing: more than €300 per night. You get 1,000 listings, in their own units this time, with text columns such as the neighborhood. As in Lab 2, 2,000 other listings stay locked away: the **test set**. At the end of the lab the tutor unlocks it, and every pipeline in the room is judged on listings it has never seen.

**Your mission.** Build one pipeline, preprocessing and model together, that classifies listings as premium or not, and estimate its accuracy on new listings. The warm-up tasks show you the pieces one by one; after that, you decide how to put them together.

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Write the code yourself, or fill in the dots `...`. A check below the cell shows an orange *Not yet* box until you are done. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

A pipeline (Lecture 6) works like every model in scikit-learn:

```python
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1))   # preprocessing, then the model
pipe.fit(X_train, y_train)          # fits the scaler, scales, fits the model
pipe.predict(X_new)                 # the class: 1 (premium) or 0 (not premium)
pipe.predict_proba(X_new)           # the probability of each class (logistic regression)
pipe.score(X_new, y_new)            # for classification: the accuracy, the share classified correctly
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1–3. Warm-up: logistic regression, scaling, text columns | together on the screen, 10 |
| 4. Your mission | 45 |
| 5. The test set is unlocked | 5 |
| 6. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, downloads the tools of this lab and the data from Inside Airbnb, and cleans them as in Lab 1. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

# The tools of this lab are in the course lab repository;
# they also download the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab03_tools.py", "lab03_tools.py")
from lab03_tools import *

X, y = load_training_data()    # your 1,000 listings: features X, and y = 1 for premium
setup_ok()

**Run.** Your data: one row per listing. `y` is the label: 1 if the listing costs more than €300 per night, 0 if not. `X` holds fourteen features, **in their own units**: unlike Lab 2, nothing has been scaled for you.

| Feature | Meaning |
|---|---|
| `accommodates`, `bedrooms`, `beds`, `bathrooms` | maximum number of guests; number of bedrooms, beds and bathrooms. The raw file has the bathrooms as text, such as `"1.5 baths"`; we turned them into numbers (Lecture 6, "Cleaning the raw survey"). |
| `minimum_nights`, `availability_365` | shortest stay allowed, nights still bookable in the coming year |
| `number_of_reviews`, `reviews_per_month`, `review_scores_rating` | number of reviews, in all and per month; average rating (1 to 5) |
| `host_listings`, `superhost` | listings of the host in Amsterdam; 1 if the host is a superhost |
| `km_to_center` | distance to Dam Square in km, as in Lab 2 |
| `room_type`, `neighbourhood` | **text**: the type of rental, and one of 22 neighborhoods |

The lists `NUMERIC` and `CATEGORICAL` hold the names of the number columns and of the two text columns.

In [ ]:
print(X.shape, "  share of premium listings:", round(y.mean(), 2))
X.head()

## Warm-up: the pieces, one by one (Tasks 1 to 3)

We go through Tasks 1 to 3 together on the screen. Just run the cells and follow along; there is nothing to fill in. Everything you see here, you will reuse in your mission.

## Task 1. Logistic regression on two features

We start with two features, so that we can draw the model: the number of bedrooms, and the distance to the center (Lecture 5, "A straight decision boundary").

**Run.** Fit a logistic regression on the two columns, and look at its coefficients.

In [ ]:
two = ["bedrooms", "km_to_center"]
logreg = LogisticRegression()
logreg.fit(X[two], y)                 # first the features, then the labels

print("coefficients:", logreg.coef_.round(2), "  intercept:", logreg.intercept_.round(2))

**Run.** Now we draw the model, as in Lecture 5. The recipe works for any classifier with two features: ask the model for the probability at every point of a fine grid, color the grid by that probability, draw the line where the probability is 0.5, and put the listings on top. You can reuse this code for your own plots.

In [ ]:
# 1. A grid of 200 x 200 points that covers the plot
bedrooms = np.linspace(-0.5, 5.5, 200)
km = np.linspace(0, 8, 200)
B, K = np.meshgrid(bedrooms, km)
grid = pd.DataFrame({"bedrooms": B.ravel(), "km_to_center": K.ravel()})

# 2. The probability of premium at every grid point, back in the shape of the grid
P = logreg.predict_proba(grid)[:, 1].reshape(B.shape)

fig, ax = plt.subplots(figsize=(7, 4.5))
# 3. Colors: the probability; a black line where the probability is 0.5
filled = ax.contourf(B, K, P, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.6)
ax.contour(B, K, P, levels=[0.5], colors="black", linewidths=2)
fig.colorbar(filled, ax=ax, label="probability of premium")

# 4. The listings on top: red circles for not premium, blue triangles for premium.
#    Bedrooms are whole numbers, so we shift every dot a little to the left or right
#    at random ("jitter"); otherwise the dots of 2 bedrooms would all lie on one line.
jitter = np.random.default_rng(0).uniform(-0.25, 0.25, len(X))
premium = (y == 1).to_numpy()
ax.scatter(X["bedrooms"][~premium] + jitter[~premium], X["km_to_center"][~premium],
           marker="o", s=12, color="#C0002A", label="not premium")
ax.scatter(X["bedrooms"][premium] + jitter[premium], X["km_to_center"][premium],
           marker="^", s=12, color="#25567B", label="premium")

ax.set_xlabel("bedrooms")
ax.set_ylabel("distance to the center (km)")
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(0, 8)
ax.legend(loc="upper right")
plt.show()

**Run.** Two listings: one bedroom, 4 km from the center; and three bedrooms, 1 km from the center.

In [ ]:
new = pd.DataFrame({"bedrooms": [1, 3], "km_to_center": [4.0, 1.0]})
print("predict_proba:")
print(logreg.predict_proba(new).round(2))      # columns: P(not premium), P(premium)
print("predict:", logreg.predict(new))

## Task 2. Does scaling matter?

Now all twelve number columns. Look at their ranges: `availability_365` runs from 0 to 365, `superhost` from 0 to 1.

In [ ]:
X[NUMERIC].describe().loc[["min", "max"]].round(1)

**Run.** $k$-NN with $k = 15$ on the twelve numbers, without and with a `StandardScaler` in front of it, each with 5-fold cross-validation. The *accuracy* is the share of listings classified correctly. `make_pipeline` glues the scaler and the model together, so that the scaler is fitted again inside every fold, on the training part only (Lecture 6, "Pipelines").

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=15)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))

print("15-NN, unscaled:", round(cross_val_score(knn_raw, X[NUMERIC], y, cv=5).mean(), 3))
print("15-NN, scaled:  ", round(cross_val_score(knn_scaled, X[NUMERIC], y, cv=5).mean(), 3))

**Run.** Three ways to scale (Lecture 6, "Three ways to scale"). Each scaler treats every column on its own. Here are two columns of your data, the distance to the center and the number of reviews, before and after each scaler. The shape of the cloud never changes: a scaler only changes the numbers on the axes. Look at those numbers, and at where the listing with more than 1,000 reviews ends up.

In [ ]:
two_cols = X[["km_to_center", "number_of_reviews"]]
scalers = [("original data", None),
           ("StandardScaler", StandardScaler()),
           ("MinMaxScaler", MinMaxScaler()),
           ("RobustScaler", RobustScaler())]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (name, scaler) in zip(axes, scalers):
    if scaler is None:
        data = two_cols.to_numpy()
    else:
        data = scaler.fit_transform(two_cols)       # learn the scale, and apply it
    ax.scatter(data[:, 0], data[:, 1], s=6)
    ax.set_title(name)
    ax.set_xlabel("distance to the center")
axes[0].set_ylabel("number of reviews")
plt.tight_layout()
plt.show()

- **`StandardScaler`**: every column gets mean 0 and standard deviation 1. **The default choice.**
- **`MinMaxScaler`**: every column runs from 0 to 1. Use it when a fixed range is needed. One outlier squeezes all the other values together: almost all the numbers of reviews end up between 0 and 0.2.
- **`RobustScaler`**: every column gets median 0, and the middle half of the values lies between −0.5 and 0.5. **Use it when there are outliers**: they stay outliers, but they do not squeeze the rest.

One scaler for all the number columns is the usual choice. Every scaler learns from the data (a mean, a minimum, a median), so it belongs inside the pipeline.

## Task 3. Text columns

`room_type` and `neighbourhood` are text, and a model can only compute with numbers. Numbering the 22 neighborhoods 1 to 22 would invent an order and distances that do not exist. One-hot encoding gives every category its own 0/1 column instead (Lecture 6, "The program column").

**Run.** What the one-hot encoder makes of the room type of the first five listings: one column per room type, with a 1 in the column of its own type.

In [ ]:
encoder = OneHotEncoder()
encoder.fit(X[["room_type"]])                    # learns the categories: the room types in the data
print(encoder.categories_)
one_hot = encoder.transform(X[["room_type"]][:5]).toarray()
pd.DataFrame(one_hot, columns=encoder.get_feature_names_out()).join(X["room_type"][:5])

**Run.** In a pipeline, a *column transformer* sends each group of columns to its own step: the numbers to the scaler, the text columns to the one-hot encoder. After that, the model sees only numbers. With `handle_unknown="ignore"`, a category that was not in the training data (a rare neighborhood in a validation fold, for example) gets all zeros instead of an error.

In [ ]:
prep = make_column_transformer(
    (StandardScaler(), NUMERIC),                             # the 12 numbers: scale
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))   # the 2 text columns: one-hot
pipe_cat = make_pipeline(prep, LogisticRegression())

print("columns the model sees:", prep.fit_transform(X).shape[1])
print("mean validation accuracy:", round(cross_val_score(pipe_cat, X, y, cv=5).mean(), 3))

## Task 4. Your mission

Now it is your turn. Build **one pipeline** that classifies listings as premium or not, and estimate its accuracy on the locked listings.

You do this in five short steps. Step 1 is ready to run. In Steps 2 to 5 you write the code yourself: the comments in each cell say what to do, and you can copy from Tasks 1 to 3, from Step 1 and from the **toolbox** right below. After every run, ask yourself: did the mean validation accuracy go up or down? Under every step you find where to look if you are stuck: a part of the toolbox, a section of the **cheat sheet**, or a slide of the lectures.

**The rule:** you may do anything with `X` and `y`. The test set stays locked.

Write down your results in the notes cell at the end of the mission; you will need them to choose.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

### 1. A pipeline: the basic structure

A pipeline is a list of steps that the data goes through, in order. Ours has two steps: first **prepare the columns**, then **the model**. The first step is a *column transformer*: it sends every group of columns to its own treatment, the numbers to a scaler and the text columns to the one-hot encoder.

```python
my_pipe = make_pipeline(
    make_column_transformer(                                   # step 1: prepare the columns
        (StandardScaler(), my_numbers),                        #   numbers: put on one scale
        (OneHotEncoder(handle_unknown="ignore"), my_text)),    #   text: one 0/1 column per category
    LogisticRegression(C=1))                                   # step 2: the model

scores = cross_val_score(my_pipe, X, y, cv=5)                  # 5-fold cross-validation, as in Lab 2
print(round(scores.mean(), 3))                                 # the mean validation accuracy
```

Cross-validation fits the whole pipeline again in every fold, so the scaler and the encoder learn only from the training part: no leak.

### 2. Choosing the columns and the scaler

**The scaler.** Three choices, as in Task 2: `StandardScaler()` (mean 0, standard deviation 1: the default), `MinMaxScaler()` (0 to 1: when you need a fixed range; outliers squeeze the rest) and `RobustScaler()` (median 0: when there are outliers). In the mission, `my_scaler` is the scaler for all your number columns:

```python
my_scaler = RobustScaler()       # one line to change; the pipeline uses my_scaler
```

Counts such as bedrooms or guests are numbers, not categories: they go to the scaler too. Only columns of categories, such as `room_type`, go to the one-hot encoder.

**The columns.** `my_numbers` and `my_text` are ordinary Python lists of column names. A column that is in neither list is left out. You can choose them yourself:

```python
my_numbers = ["accommodates", "bedrooms", "bathrooms", "km_to_center"]   # only these four numbers
my_text = ["room_type"]                                                   # only the room type
my_text = []                                                              # no text columns at all
```

More columns is not always better: a column that says nothing new adds noise.

### 3. The penalty `C`, and how to choose it

Logistic regression keeps its weights small with a penalty. `C` sets how strong it is, the other way round from `alpha` in Lab 2:

- a **small** `C` (0.001): a strong penalty, all weights close to 0, a model that is too simple (underfitting);
- a **large** `C` (100): a weak penalty, a model that can follow the training data too closely (overfitting).

Do not guess `C`: let a **grid search** try a list of values with cross-validation and keep the best, as in Lab 2. Inside a pipeline, the setting is called by the name of its step in lower case, two underscores, and its own name: `logisticregression__C`.

```python
my_Cs = [0.001, 0.01, 0.1, 1, 10, 100]
grid = GridSearchCV(my_pipe, {"logisticregression__C": my_Cs}, cv=5)
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 3))   # the best C and its mean validation accuracy
grid.best_estimator_                                   # the pipeline with the best C, refitted on all of X, y
```

### 4. Another model

Change only the last step of the pipeline; the column transformer stays the same.

```python
LinearSVC(C=1)                          # a linear SVM: also a straight boundary, the same C, no probabilities
KNeighborsClassifier(n_neighbors=15)    # k-NN: the k most similar listings vote
```

Their settings in a grid search: `"linearsvc__C"` and `"kneighborsclassifier__n_neighbors"`.

### 5. The logarithm of a long tail

Some columns have a **long tail**: most values are small and a few are huge, such as the number of reviews. The few huge values pull a linear model around. The logarithm pulls the tail in: from 10 to 20 reviews then counts as much as from 100 to 200. `np.log1p(x)` computes $\log(1 + x)$, which also works for 0.

Give the long-tailed columns their own line in the column transformer: first the log, then the scaler.

```python
long_tail = ["number_of_reviews", "minimum_nights"]          # your long-tailed columns
other = [c for c in my_numbers if c not in long_tail]        # all the other numbers

my_pipe = make_pipeline(
    make_column_transformer(
        (make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), long_tail),   # log, then scale
        (StandardScaler(), other),                                                     # scale
        (OneHotEncoder(handle_unknown="ignore"), my_text)),                           # one-hot
    LogisticRegression(C=1))
```

### 6. Extra: interactions and squares

A linear model draws a straight boundary. Products of two columns (*interactions*) and squares let it bend. `PolynomialFeatures(degree=2)` adds every square and every product of two columns: $[a, b]$ becomes $[1, a, b, a^2, ab, b^2]$. With twelve numbers that gives 91 columns, so let `C` keep them in check, and expect this to help only a little here.

```python
my_pipe = make_pipeline(
    make_column_transformer(
        (make_pipeline(StandardScaler(), PolynomialFeatures(degree=2)), my_numbers),   # scale, then products and squares
        (OneHotEncoder(handle_unknown="ignore"), my_text)),
    LogisticRegression(C=0.1, max_iter=2000))
```

### 7. Putting it together: a different treatment for every group of columns

The column transformer is the heart of every pipeline in this lab: it splits your table into **groups of columns**, gives every group **its own treatment**, and glues the results back together, side by side, into one table of numbers for the model. A treatment can be one step (a scaler), or a small pipeline of several steps (first the log, then the scaler). Below, five groups and five treatments in one pipeline: you can copy it and change the groups.

```python
# 1. Make the groups: ordinary lists of column names.
size = ["accommodates", "bedrooms", "bathrooms"]                      # counts: just scale
long_tail = ["number_of_reviews", "reviews_per_month", "minimum_nights"]   # long tails: log, then scale
location = ["km_to_center", "review_scores_rating"]                   # these two: also their product

# 2. A feature from knowledge of the problem: bathrooms per bedroom.
def baths_per_bedroom(d):                                            # d: the two columns given below
    return d[["bathrooms"]].to_numpy() / np.maximum(d[["bedrooms"]].to_numpy(), 1)   # a studio counts as 1 bedroom

# 3. One treatment per group. Each line: (what to do, which columns).
prep = make_column_transformer(
    (StandardScaler(), size),
    (make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), long_tail),
    (make_pipeline(StandardScaler(),
                   PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)), location),
    (make_pipeline(FunctionTransformer(baths_per_bedroom), StandardScaler()), ["bathrooms", "bedrooms"]),
    (OneHotEncoder(handle_unknown="ignore"), ["room_type"]))

# 4. The pipeline: the column transformer first, then the model.
pipe = make_pipeline(prep, LogisticRegression(C=0.1))

print("columns the model sees:", prep.fit_transform(X).shape[1])
print("mean validation accuracy:", round(cross_val_score(pipe, X, y, cv=5).mean(), 3))
```

What every line does:

- **`StandardScaler(), size`**: the three counts are only scaled. Counts are numbers, not categories: they go to a scaler, never to the one-hot encoder.
- **`make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), long_tail`**: first the logarithm pulls the long tail in, then the scaler puts the result on one scale. A small pipeline *inside* the column transformer runs its steps in order.
- **`PolynomialFeatures(degree=2, interaction_only=True, include_bias=False), location`**: keeps the two columns and adds their **product**, `km_to_center × review_scores_rating`: a well-rated listing may be worth more close to the center than far away. `interaction_only=True` gives only products, no squares; `include_bias=False` leaves out the constant column of ones (the model has its own intercept). On a small group you get one new column, instead of 91 on all twelve numbers.
- **`FunctionTransformer(baths_per_bedroom)`**: your own feature, from *knowledge of the problem* (Lecture 6, "Features from domain knowledge"). Because it sits inside the pipeline, it is computed again in every fold, and cross-validation stays honest.
- **`OneHotEncoder(handle_unknown="ignore"), ["room_type"]`**: the text column becomes 0/1 columns, one per room type. One-hot columns are not scaled.
- **Columns in no group are left out:** here `neighbourhood`, `beds` and a few more. To keep them unchanged instead, add `remainder="passthrough"` as the last argument of `make_column_transformer`.
- **A column may be in two groups:** `bedrooms` is scaled in `size` and also used for the ratio. Both versions go to the model.

Everything is still one pipeline: `cross_val_score(pipe, X, y, cv=5)`, `GridSearchCV(pipe, {"logisticregression__C": [...]}, cv=5)` and `pipe.fit(X, y)` work exactly as before. More treatments do not automatically mean a better score: compare with cross-validation, one change at a time.

</details>

### Step 1. The starting pipeline

**Run.** This is the pipeline of Task 3, written out step by step. It is your starting point: in every later step you copy it and change one thing.

In [ ]:
# Step 1. Run this cell. Nothing to change.
my_numbers = NUMERIC          # the 12 number columns
my_text = CATEGORICAL         # the 2 text columns: room_type and neighbourhood

my_pipe = make_pipeline(
    make_column_transformer(
        (StandardScaler(), my_numbers),
        (OneHotEncoder(handle_unknown="ignore"), my_text)),
    LogisticRegression(C=1))

scores = cross_val_score(my_pipe, X, y, cv=5)
print("Step 1, starting pipeline: mean validation accuracy", round(scores.mean(), 3))

*Stuck?* Toolbox, part 1 · Cheat sheet, section 5 "Pipelines and cross-validation" · Lecture 6, "Pipelines".

### Step 2. Choose your columns and your scaler

Do all fourteen columns help, or do some only add noise? Make your own lists of columns. And which of the three scalers of Task 2 works best for your columns?

In [ ]:
# Step 2. We compare the three scalers, and all columns with the same pipeline without the neighbourhood.
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    pipe = make_pipeline(
        make_column_transformer(
            (scaler, NUMERIC),
            (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)),
        LogisticRegression(C=1))
    print(f"Step 2, {type(scaler).__name__}: mean validation accuracy", round(cross_val_score(pipe, X, y, cv=5).mean(), 3))
my_scaler = StandardScaler()       # the three are within the noise between folds: we keep the default

my_numbers = ["accommodates", "bedrooms", "beds", "bathrooms", "minimum_nights", "availability_365",
              "number_of_reviews", "reviews_per_month", "review_scores_rating", "host_listings",
              "superhost", "km_to_center"]
my_text = ["room_type", "neighbourhood"]

for name, text in [("all columns", my_text), ("without neighbourhood", ["room_type"])]:
    pipe = make_pipeline(
        make_column_transformer(
            (my_scaler, my_numbers),
            (OneHotEncoder(handle_unknown="ignore"), text)),
        LogisticRegression(C=1))
    print(f"Step 2, {name}: mean validation accuracy", round(cross_val_score(pipe, X, y, cv=5).mean(), 3))

# Leaving out the neighbourhood helps a little: its 22 one-hot columns mostly add noise,
# and km_to_center already says where a listing is. We continue without it.
my_text = ["room_type"]
my_pipe = make_pipeline(
    make_column_transformer(
        (my_scaler, my_numbers),
        (OneHotEncoder(handle_unknown="ignore"), my_text)),
    LogisticRegression(C=1))

*Stuck?* Toolbox, part 2 · Cheat sheet, sections 4 "Scaling" and 6 "Text columns" · Lecture 6, "Three ways to scale" and "Numbers and categories together".

### Step 3. Choose `C` with a grid search

`C` is the penalty of logistic regression: small `C`, simple model; large `C`, close fit. Do not guess it: let a grid search try a list of values with 5-fold cross-validation, on your pipeline of Step 2, as in Lab 2.

In [ ]:
# Step 3. A wide range of C, from far too strong to far too weak a penalty.
my_Cs = [0.00001, 0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]

grid = GridSearchCV(my_pipe, {"logisticregression__C": my_Cs}, cv=5)
grid.fit(X, y)
print("Step 3, best C:", grid.best_params_["logisticregression__C"],
      "  mean validation accuracy", round(grid.best_score_, 3))

fig, ax = plt.subplots()
ax.plot([str(c) for c in my_Cs], grid.cv_results_["mean_test_score"], marker="o")
ax.set_xlabel("C (larger = weaker penalty)")
ax.set_ylabel("mean validation accuracy")
ax.tick_params(axis="x", rotation=90)
plt.show()

**What the plot shows.** With `C = 0.00001` the penalty pushes every weight to 0, and the model predicts "not premium" for every listing: about 0.52, the baseline (underfitting). As `C` grows, the accuracy climbs to about 0.79 at `C = 0.03`, and then stays flat all the way to 100: with 1,000 listings and only 15 columns, a weak penalty overfits hardly at all. On such a flat top the exact "best" `C` is luck of the folds; any value from 0.03 up is as good.

*Stuck?* Toolbox, part 3 · Cheat sheet, section 2 "The penalty C" and the grid search in section 5 · Lecture 5, "The penalty C" and "Choosing C".

### Step 4. Try another model

Keep your columns, and change only the model at the end of the pipeline: a linear SVM or $k$-NN.

In [ ]:
# Step 4. Both models on the same columns, each with its setting chosen by a grid search.
for model, setting, values in [(KNeighborsClassifier(), "kneighborsclassifier__n_neighbors", list(range(1, 61))),
                               (LinearSVC(), "linearsvc__C", [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 10])]:
    pipe = make_pipeline(
        make_column_transformer(
            (my_scaler, my_numbers),
            (OneHotEncoder(handle_unknown="ignore"), my_text)),
        model)
    g = GridSearchCV(pipe, {setting: values}, cv=5).fit(X, y)
    print(f"Step 4, {type(model).__name__}: best {setting.split('__')[1]} = {g.best_params_[setting]},",
          "mean validation accuracy", round(g.best_score_, 3))
# k-NN stays a little lower; the linear SVM ends at the level of logistic regression.

*Stuck?* Toolbox, part 4 · Cheat sheet, section 3 "Linear SVM" · Lecture 5, "Logistic regression or linear SVM?".

### Step 5. Better features: the logarithm of a long tail

First look at the shape of every number column. A column with a **long tail** has most of its values on the left and a few far to the right. (Columns such as bedrooms or guests look like separate bars because they are whole numbers: they are counts, not categories, and stay with the other numbers.)

In [ ]:
# Step 5a. Run this cell. Which columns have a long tail?
X[my_numbers].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
# Step 5b. The four count columns have a long tail.
long_tail = ["minimum_nights", "number_of_reviews", "reviews_per_month", "host_listings"]
other = [c for c in my_numbers if c not in long_tail]

my_pipe3 = make_pipeline(
    make_column_transformer(
        (make_pipeline(FunctionTransformer(np.log1p), my_scaler), long_tail),   # log, then scale
        (my_scaler, other),
        (OneHotEncoder(handle_unknown="ignore"), my_text)),
    LogisticRegression(C=grid.best_params_["logisticregression__C"]))

scores = cross_val_score(my_pipe3, X, y, cv=5)
print("Step 5, with the log: mean validation accuracy", round(scores.mean(), 3))
# A small gain at most: the differences between the best pipelines are about one hundredth,
# the size of the differences between folds.

*Stuck?* Toolbox, part 5 (part 6 for products and squares, part 7 for everything together) · Cheat sheet, section 7 "New features" · Lecture 6, "A long tail" and "Interactions".

### Your notes

**Your turn.** Double-click the cell below and write down, for each step, what you tried and its mean validation accuracy. Then choose your best pipeline.

**Answer.** Step 1, starting pipeline: about 0.77. Step 2, without the neighbourhood: about 0.79, a little better. Step 3, `C`: a flat top from 0.03 up, about 0.79. Step 4, $k$-NN about 0.77, linear SVM about 0.79. Step 5, logarithm of the four count columns: about 0.79. Our choice: the pipeline of Step 3, with an estimate of 0.79. Steps 3, 4 and 5 differ by less than one hundredth, the noise between folds: any of them is a good choice.

### Your submission

**Your turn.** Fill in your pipeline, a few words on how you chose it, and the accuracy you expect on the locked listings (your best mean validation accuracy is a good estimate). The last line fits your pipeline once more on **all** 1,000 training listings.

In [ ]:
my_pipeline = grid.best_estimator_     # Step 3: without the neighbourhood, C by a 5-fold grid search
my_strategy = "without the neighbourhood, C chosen by a 5-fold grid search"
my_estimate = round(grid.best_score_, 2)

my_pipeline.fit(X, y)  # learn from all 1,000 training listings

In [ ]:
# Check
check_submission()

**Think.** Before the test set is unlocked: will your pipeline score higher or lower on it than your estimate? Why?

**Answer.** Answers differ by strategy. A cross-validation estimate should be close, but not exact: each fold's model learns from only four fifths of the listings, which makes the estimate a little pessimistic, and picking the best of many settings makes it a little optimistic. With 2,000 test listings, the test accuracy itself is precise to about one or two hundredths; differences between pipelines smaller than that are luck.

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare your pipeline with your neighbor's: the same model? The same preprocessing?

## Task 5. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it scores your pipeline on the 2,000 listings that no model has seen.

In [ ]:
X_test, y_test = unlock_test_set("straightline")    # the password on the screen

In [ ]:
test_accuracy = report(my_pipeline, my_estimate, X_test, y_test)

Tell the tutor your pipeline, how you chose it, your estimate and your test accuracy. We collect them on the screen.

> **As in Lab 2:** the test set has now been used. Changing your pipeline to do better on it would make its score an optimistic estimate.

## Task 6. What happened?

**Run.** The pieces of today, one by one: for each pipeline, the mean 5-fold cross-validation accuracy on your 1,000 listings, and the accuracy on the test set. The first row always predicts the most common class.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
from sklearn.dummy import DummyClassifier
long_tail = ["minimum_nights", "number_of_reviews", "reviews_per_month", "host_listings"]
other = [c for c in NUMERIC if c not in long_tail]
numbers = make_column_transformer((StandardScaler(), NUMERIC))
numbers_raw = make_column_transformer(("passthrough", NUMERIC))
with_text = make_column_transformer((StandardScaler(), NUMERIC),
                                    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
with_log = make_column_transformer((make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), long_tail),
                                   (StandardScaler(), other),
                                   (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
candidates = {
    "always the most common class": DummyClassifier(),
    "15-NN, numbers, unscaled": make_pipeline(numbers_raw, KNeighborsClassifier(15)),
    "15-NN, numbers, scaled": make_pipeline(numbers, KNeighborsClassifier(15)),
    "logistic regression, numbers, scaled": make_pipeline(numbers, LogisticRegression()),
    "  + text columns": make_pipeline(with_text, LogisticRegression()),
    "  + log of the long tails": make_pipeline(with_log, LogisticRegression()),
    "  same, C = 0.00001": make_pipeline(with_log, LogisticRegression(C=0.00001)),
    "linear SVM, same features": make_pipeline(with_log, LinearSVC()),
}
rows = []
for name, m in candidates.items():
    rows.append({"pipeline": name,
                 "5-fold CV": cross_val_score(m, X, y, cv=5).mean(),
                 "test set": m.fit(X, y).score(X_test, y_test)})
pd.DataFrame(rows).set_index("pipeline").round(3)

**Think.** We discuss these together.

(a) How much did scaling give $k$-NN? Would you ever leave it out of a pipeline with $k$-NN?<br>
(b) Did the text columns help, by cross-validation? On the test set? Which of the two do you trust to decide, and why?<br>
(c) What does logistic regression with `C = 0.00001` do? Compare it with the first row.<br>
(d) Logistic regression or linear SVM: does the choice matter much here? When would you need logistic regression? (Lecture 5, "Logistic regression or linear SVM?")<br>
(e) We dropped every listing with a missing value, about one in six. Who might those listings be, and could that matter? (Lecture 7 is about this.)

**Answer.** (a) More than ten points of accuracy: from about 0.63, not far above always guessing, to about 0.76, close to logistic regression. In a pipeline with $k$-NN, scaling should always be there: it never hurts a distance-based model (Lecture 6). (b) By cross-validation the text columns change nothing; on the test set they look better, by a couple of points. Decide with cross-validation: it averages five splits, while one test set is a single draw, and once you use the test set to choose, it is no longer an honest estimate. (c) Its penalty is so strong that all coefficients are almost 0, so it predicts the most common class for every listing: the same accuracy as the first row. Underfitting. (d) Their accuracy is almost the same here (Lecture 5: "their accuracy is often similar"). If you need a probability, for example to rank listings by how likely they are premium, use logistic regression: `LinearSVC` has no `predict_proba`. (e) Mostly listings without reviews: new listings, or listings that are rarely booked. If they differ from the others in price, a model trained without them may do worse on them. Dropping is the simplest fix, not always the best; Lecture 7 shows how to fill the gaps instead.

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** For a listing, `predict_proba` returns `[0.35, 0.65]`. What does `predict` return?

A. 0 (not premium) &nbsp; B. 1 (premium) &nbsp; C. 0.65 &nbsp; D. It depends on `C`

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The second column is the probability of class 1, premium. It is above 0.5, so `predict` says premium (Lecture 5, "Logistic regression in scikit-learn").

</details>

**2.** A colleague fits a `StandardScaler` on all 1,000 listings, and then runs 5-fold cross-validation on the scaled data. What is wrong?

A. Nothing &nbsp; B. The validation folds helped to fit the scaler: a leak &nbsp; C. The scaler should be fitted on the test set &nbsp; D. Logistic regression needs no scaling

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The validation fold shaped the mean and the standard deviation before it was used to validate. Put the scaler in a pipeline, and it is refitted inside every fold, on the training part only (Lecture 6, "Where does the scaler go in cross-validation?").

</details>

**3.** `neighbourhood` has 22 values. How many columns does `OneHotEncoder` make of it?

A. 1 &nbsp; B. 2 &nbsp; C. 21 &nbsp; D. 22

<details><summary><b>Answer</b> (click to unfold)</summary>

**D.** One 0/1 column per neighborhood. (Statisticians drop one column; with a penalized model we keep all of them, Lecture 6, "The program column".)

</details>

**4.** Unscaled, which feature has the largest influence on the distances of $k$-NN in this lab?

A. `superhost` &nbsp; B. `bathrooms` &nbsp; C. `availability_365` &nbsp; D. `km_to_center`

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** It runs from 0 to 365, so its differences are by far the largest numbers (Lecture 3, "Distances depend on the units"; Lecture 6, "Two debts from earlier lectures").

</details>

**5.** A hosting platform wants to show each host *the probability* that their listing counts as premium. Which model?

A. `LinearSVC` &nbsp; B. `LogisticRegression` &nbsp; C. $k$-NN with $k = 1$ &nbsp; D. Any of them

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Logistic regression turns its score into a probability; a linear SVM gives only a class (Lecture 5, "Logistic regression or linear SVM?"). 1-NN gives probabilities of only 0 or 1.

</details>

## If you finish early

**A. Which features does the L1 penalty keep?** Fit `make_pipeline(prep, LogisticRegression(penalty="l1", solver="liblinear", C=0.05))` on `X, y`. The names of the columns after the preprocessing are in `pipe[0].get_feature_names_out()`, the coefficients in `pipe[-1].coef_[0]`. Which features survive? Try a few values of `C`.

In [ ]:
l1 = make_pipeline(prep, LogisticRegression(penalty="l1", solver="liblinear", C=0.05)).fit(X, y)
coefs = pd.Series(l1[-1].coef_[0], index=l1[0].get_feature_names_out())
print((coefs != 0).sum(), "of", len(coefs), "features kept")
print(coefs[coefs != 0].round(2).sort_values())
# About a third of the 37 columns survive: the bedrooms, guests and bathrooms, the distance
# to the center, the reviews per month, "private room" and the host. Every neighborhood is set to 0:
# the distance to the center already says what they would add. A larger C keeps more features.

**B. Three classes.** `load_price_classes()` gives the same listings in three classes: budget (up to €200), middle (up to €350) and luxury. Fit `make_pipeline(prep, LogisticRegression())` on `X` and these labels, and look at `predict_proba` for the first five listings. How many columns, and what do they sum to? (Lecture 5, "Multiclass in scikit-learn".)

In [ ]:
y3 = load_price_classes()
three = make_pipeline(prep, LogisticRegression()).fit(X, y3)
print(three.classes_)
print(three.predict_proba(X[:5]).round(2), three.predict(X[:5]))
print("mean validation accuracy:", round(cross_val_score(three, X, y3, cv=5).mean(), 3))
# Three columns, one per class, in the order of classes_ (alphabetical); each row sums to 1.
# The same code as for two classes: nothing to change.

## Where this is going

> Put every step that learns from data inside the pipeline: then cross-validation judges the preprocessing and the model together, honestly.

Today the preprocessing mattered more than the choice between linear models: scaling for $k$-NN, the logarithm for the long tails, and a penalty strong enough, but not too strong.

**Coming up.**

- Lecture 7 (Monday, November 16): missing values and feature selection. Today we dropped the listings with gaps; next week you fill them, inside the pipeline.
- Lecture 8 (Tuesday, November 17): decision trees and random forests, models that need no scaling at all.

**Reading.** Müller and Guido (2016), Chapter 2: "Linear models for classification"; Chapter 3: "Preprocessing and Scaling"; Chapter 4: "Categorical Variables" and "Univariate Nonlinear Transformations"; Chapter 6: "Building Pipelines".

---
*This lab practices Lectures 5 and 6 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website and cleaned as in Lab 1.*